# Climate Data Cleaning: ISU Climate Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from shapely.geometry import Point
from pathlib import Path

In [2]:
data_path = r'../../../../data/tabular/climate/raw'
df_climate = pd.read_csv(f'{data_path}/isu-climate.csv')

In [3]:
# Step 1: Inspect
print(df_climate.shape)
df_climate.head()

(115602, 12)


,station,station_name,day,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,IA1533,CLARINDA,2024-01-01,1,0.0,29.0,-1.7,21.0,-6.1,0.0000,0.0000,0.0000
1,IA0807,Boone,2024-01-01,1,0.0,29.0,-1.7,20.0,-6.7,0.0001,0.0001,0.0000
2,IA2689,Emmetsburg,2024-01-01,1,0.0,29.0,-1.7,20.0,-6.7,0.0000,NaN,NaN
3,IA6199,OELWEIN 1E,2024-01-01,1,0.0,31.0,-0.6,21.0,-6.1,0.0001,0.0001,0.0001
4,IAC003,Iowa - Northeast Climate Division,2024-01-01,1,0.0,32.0,0.0,23.0,-5.0,0.0000,0.0000,0.0000


In [4]:
# Step 2: Fix Data Types
df_climate['day'] = pd.to_datetime(df_climate['day'], errors='coerce')

num_cols = ['doy', 'gdd_40_86', 'high', 'highc', 'low', 'lowc', 'precip', 'snow', 'snowd']
for col in num_cols:
    df_climate[col] = pd.to_numeric(df_climate[col], errors='coerce')

# Step 3: Handle Missing Values
print("Missing values per column:")
print(df_climate.isnull().sum())

# Snow and snow depth are missing when there was no snow - fill with 0
df_climate['snow'] = df_climate['snow'].fillna(0)
df_climate['snowd'] = df_climate['snowd'].fillna(0)

# Drop rows missing temperature or precipitation (critical measurements)
df_climate.dropna(subset=['high', 'low', 'precip'], inplace=True)

print(f"\nRows remaining: {len(df_climate):,}")

Missing values per column:
station             0
station_name        0
day                 0
doy                 0
gdd_40_86           0
high             1226
highc            1226
low              1549
lowc             1549
precip            386
snow            12815
snowd           14694
dtype: int64

Rows remaining: 114,052


In [5]:
print(f"shape: {df_climate.shape}")
print(f"columns: {df_climate.columns.tolist()}")

shape: (114052, 12)
columns: ['station', 'station_name', 'day', 'doy', 'gdd_40_86', 'high', 'highc', 'low', 'lowc', 'precip', 'snow', 'snowd']


In [6]:
df_climate.head()

,station,station_name,day,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,IA1533,CLARINDA,2024-01-01,1,0.0,29.0,-1.7,21.0,-6.1,0.0000,0.0000,0.0000
1,IA0807,Boone,2024-01-01,1,0.0,29.0,-1.7,20.0,-6.7,0.0001,0.0001,0.0000
2,IA2689,Emmetsburg,2024-01-01,1,0.0,29.0,-1.7,20.0,-6.7,0.0000,0.0000,0.0000
3,IA6199,OELWEIN 1E,2024-01-01,1,0.0,31.0,-0.6,21.0,-6.1,0.0001,0.0001,0.0001
4,IAC003,Iowa - Northeast Climate Division,2024-01-01,1,0.0,32.0,0.0,23.0,-5.0,0.0000,0.0000,0.0000


In [7]:
# Step 4: Save Cleaned Data
import os
out_path = '../../../../data/tabular/climate/clean'
os.makedirs(out_path, exist_ok=True)
df_climate.to_csv(f'{out_path}/isu-climate-clean.csv', index=False)
print(f"Saved {len(df_climate):,} rows → {out_path}/isu-climate-clean.csv")

Saved 114,052 rows → ../../../../data/tabular/climate/clean/isu-climate-clean.csv
